In [6]:
"""
validate_story_1.py
Kiem tra cac gia thuyet trong Story 1 voi data thuc te.
"""

import pandas as pd
import numpy as np
import os

# ── ÉP PANDAS KHÔNG ĐƯỢC CHE GIẤU DỮ LIỆU ───────────────────────
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

root_dir      = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
processed_dir = os.path.join(root_dir, "data", "processed")

def validate_story_1():
    print("Loading data...")

    # ── 1. LOAD ───────────────────────────────────────────────────
    orders      = pd.read_parquet(os.path.join(processed_dir, "orders.parquet"))
    order_items = pd.read_parquet(os.path.join(processed_dir, "order_items.parquet"))
    geography   = pd.read_parquet(os.path.join(processed_dir, "geography.parquet"))
    sales       = pd.read_parquet(os.path.join(processed_dir, "sales.parquet"))

    # ── 2. REVENUE TU SALES (official) ───────────────────────────
    sales["Year"] = pd.to_datetime(sales["Date"], unit="ms").dt.year
    yearly_revenue = (
        sales.groupby("Year")["Revenue"]
        .sum()
        .reset_index()
    )

    # ── 3. FILTER ORDERS ──────────────────────────────────────────
    # valid_statuses = ["delivered", "shipped", "paid"]
    # orders = orders[orders["order_status"].str.lower().isin(valid_statuses)].copy()

    # ── 4. ORDER ITEMS ────────────────────────────────────────────
    order_items["discount_amount"] = order_items["discount_amount"].fillna(0)
    order_items["has_promo"] = (
        order_items["promo_id"].notna() |
        (order_items["discount_amount"] > 0)
    )

    order_agg = (
        order_items.groupby("order_id")
        .agg(
            used_promo=("has_promo", "max")
        )
        .reset_index()
    )

    # ── 5. JOIN ───────────────────────────────────────────────────
    orders["Year"]  = orders["order_date"].dt.year
    orders["Month"] = orders["order_date"].dt.month
    orders["Quarter"] = orders["Month"].map(
        lambda m: "Q2 Peak" if m in [4,5,6]
                  else "Q4 Low" if m in [10,11,12]
                  else "Other"
    )

    geo = geography[["zip","region"]].drop_duplicates("zip")

    df = (
        orders
        .merge(order_agg, on="order_id", how="left")
        .merge(geo,       on="zip",      how="left")
    )

    # ── 6. YEARLY METRICS ─────────────────────────────────────────
    yearly_orders = (
        df.groupby("Year")
        .agg(
            customers =("customer_id", "nunique"),
            orders_cnt=("order_id",    "nunique"),
        )
        .reset_index()
    )

    yearly = yearly_orders.merge(yearly_revenue, on="Year", how="left")
    yearly["opc"] = yearly["orders_cnt"] / yearly["customers"]
    yearly["aov"] = yearly["Revenue"]    / yearly["orders_cnt"]

    ym = yearly.set_index("Year")

    def yoy(col, yr, prev_yr):
        try:
            return (ym.loc[yr, col] - ym.loc[prev_yr, col]) / ym.loc[prev_yr, col] * 100
        except KeyError:
            return np.nan

    # ==============================================================
    # ── GOM TOÀN BỘ OUTPUT VÀO STRING ĐỂ LƯU FILE KHÔNG BỊ MẤT ────
    # ==============================================================
    output_str = []

    # ── 7. GIA THUYET 1 ───────────────────────────────────────────
    output_str.append("="*65)
    output_str.append("  GIA THUYET 1: CU SOC 2019 & DA PHUC HOI 2022")
    output_str.append("="*65)

    key_years  = [2017, 2018, 2019, 2020, 2021, 2022]
    prev_years = {2017:2016, 2018: 2017, 2019: 2018, 2020:2019, 2021: 2020, 2022: 2021}

    rows = []
    for yr in key_years:
        if yr not in ym.index: continue
        py = prev_years[yr]
        rows.append({
            "Year"            : yr,
            "Customers"       : int(ym.loc[yr, "customers"]),
            "Cust_YoY(%)"     : round(yoy("customers",  yr, py), 1),
            "Orders/Customer" : round(ym.loc[yr, "opc"], 2),
            "OPC_YoY(%)"      : round(yoy("opc",        yr, py), 1),
            "AOV"             : round(ym.loc[yr, "aov"]),
            "AOV_YoY(%)"      : round(yoy("aov",        yr, py), 1),
        })

    output_str.append(pd.DataFrame(rows).to_string(index=False))
    output_str.append("\n  Giai thich:\n  - Cust_YoY: % thay doi so khach active\n  - OPC_YoY: % thay doi orders/customer\n  - AOV_YoY: % thay doi gia tri TB don hang\n")

    # ── 8. GIA THUYET 2 ───────────────────────────────────────────
    output_str.append("="*65)
    output_str.append("  GIA THUYET 2: SUC DE KHANG VUNG (2018 → 2020)")
    output_str.append("="*65)

    df_reg = df[df["Year"].isin([2018, 2019, 2020])]
    reg = (
        df_reg.groupby(["Year","region"])
        .agg(
            customers =("customer_id", "nunique"),
            orders_cnt=("order_id",    "nunique")
        )
        .reset_index()
    )
    reg["opc"] = reg["orders_cnt"] / reg["customers"]

    pivot = reg.pivot(index="region", columns="Year", values="opc")
    pivot["Drop_18_20(%)"] = ((pivot[2020] - pivot[2018]) / pivot[2018] * 100).round(1)
    pivot["Drop_18_19(%)"] = ((pivot[2019] - pivot[2018]) / pivot[2018] * 100).round(1)
    
    # Gắn cờ max_rows=None để không bao giờ bị dính "..."
    output_str.append(pivot.round(3).to_string(max_rows=None))
    output_str.append("")

    # ── 9. GIA THUYET 3 ───────────────────────────────────────────
    output_str.append("="*65)
    output_str.append("  GIA THUYET 3: PROMO Q4 vs Q2 (2018-2019)")
    output_str.append("="*65)

    q = df[
        df["Year"].isin([2018, 2019]) &
        df["Quarter"].isin(["Q2 Peak", "Q4 Low"])
    ]
    promo = (
        q.groupby(["Year","Quarter"])
        .agg(
            total_orders=("order_id",   "nunique"),
            promo_orders=("used_promo", "sum")
        )
        .reset_index()
    )
    promo["penetration(%)"] = (promo["promo_orders"] / promo["total_orders"] * 100).round(1)
    
    output_str.append(promo.to_string(index=False, max_rows=None))
    output_str.append("\n  Insight: Q4 co promo cao hon nhung demand thap hon, xac nhan van de la demand.\n")

    # ── 10. SUMMARY ───────────────────────────────────────────────
    output_str.append("="*65)
    output_str.append("  SO LIEU DE DUNG TRONG PHAN TICH (official revenue)")
    output_str.append("="*65)
    output_str.append(yearly[["Year","customers","opc","aov","Revenue"]].to_string(index=False, max_rows=None))

    # ── IN RA MÀN HÌNH VÀ LƯU RA FILE TEXT ────────────────────────
    final_text = "\n".join(output_str)
    
    # 1. In ra console
    print(final_text)
    
    # 2. Lưu ra file để backup (đảm bảo ko bị terminal nuốt mất)
    report_path = os.path.join(root_dir, "reports", "story_1_validation.txt")
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(final_text)
        
    print(f"\n✅ Đã lưu kết quả full không bị cắt xén ra file: {report_path}")

validate_story_1()

Loading data...
  GIA THUYET 1: CU SOC 2019 & DA PHUC HOI 2022
 Year  Customers  Cust_YoY(%)  Orders/Customer  OPC_YoY(%)   AOV  AOV_YoY(%)
 2017      39651         -3.1             1.92        -4.6 25144        -1.7
 2018      37922         -4.4             1.83        -4.4 26617         5.9
 2019      27312        -28.0             1.52       -16.9 27326         2.7
 2020      24335        -10.9             1.43        -5.9 30232        10.6
 2021      23984         -1.4             1.44         0.4 30211        -0.1
 2022      24696          3.0             1.46         1.3 32489         7.5

  Giai thich:
  - Cust_YoY: % thay doi so khach active
  - OPC_YoY: % thay doi orders/customer
  - AOV_YoY: % thay doi gia tri TB don hang

  GIA THUYET 2: SUC DE KHANG VUNG (2018 → 2020)
Year      2018   2019   2020  Drop_18_20(%)  Drop_18_19(%)
region                                                    
Central  1.786  1.481  1.409          -21.1          -17.1
East     1.690  1.448  1.376    